# VisionGuard AI — Run the Web Application on Kaggle

Runs the full project (FastAPI backend + React frontend) and publishes it through a temporary public link.

**Before you start:** set **Internet = On** in the notebook settings (GPU optional).
Run the cells in order and open the link printed by the last cell. Keep the notebook running while you use the app.

## 1. Configuration

In [ ]:
# Change this to your own repository once the project is on your GitHub
REPO_URL = "https://github.com/AhmedRahim01/VisionGuard-AI-Industrial-Defect-Inspection.git"

WORK_DIR = "/kaggle/working"
PROJECT = f"{WORK_DIR}/visionguard"
PORT = 8000

## 2. Download the project

In [ ]:
%%bash -s "$REPO_URL" "$PROJECT"
rm -rf "$2"
git clone -q "$1" "$2"
ls "$2/backend/model"

## 3. Install dependencies

In [ ]:
%%bash -s "$PROJECT"
set -e
pip install -q -r "$1/backend/requirements.txt"
pip install -q reportlab python-dotenv email-validator python-multipart opencv-python-headless

# Node.js (needed to build the React frontend)
curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
apt-get install -y -qq nodejs > /dev/null 2>&1
node -v

## 4. Build the frontend

In [ ]:
%%bash -s "$PROJECT"
set -e
cd "$1/frontend"

# Older copies of the project hard-code the API address; make it relative so one public link serves everything
grep -rl "http://127.0.0.1:8000" src 2>/dev/null | xargs -r sed -i 's#http://127.0.0.1:8000##g'

npm install --silent
VITE_API_URL="" npm run build
ls dist

## 5. Start the application

In [ ]:
import os, subprocess, time, urllib.request

backend_dir = f"{PROJECT}/backend"

# Launcher: the API plus the built frontend on the same address
with open(f"{backend_dir}/run_app.py", "w") as f:
    f.write("""
from main import app
from fastapi.staticfiles import StaticFiles

# Replace the API's status route at "/" with the web interface
app.router.routes = [r for r in app.router.routes if getattr(r, "path", None) != "/"]
app.mount("/", StaticFiles(directory="../frontend/dist", html=True), name="ui")
""")

subprocess.run("pkill -f 'uvicorn run_app' || true", shell=True)
log = open(f"{backend_dir}/server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "run_app:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=backend_dir, stdout=log, stderr=subprocess.STDOUT,
)

for _ in range(90):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        print("Server is up.")
        break
    except Exception:
        if server.poll() is not None:
            break
        time.sleep(2)
else:
    print("Server did not respond in time.")

if server.poll() is not None:
    print("Server stopped. Last log lines:\n")
    print(open(f"{backend_dir}/server.log").read()[-2500:])

## 6. Open the public link

In [ ]:
import re, subprocess, time

subprocess.run(
    f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {WORK_DIR}/cloudflared && chmod +x {WORK_DIR}/cloudflared",
    shell=True, check=True,
)

tunnel_log = open(f"{WORK_DIR}/tunnel.log", "w")
tunnel = subprocess.Popen(
    [f"{WORK_DIR}/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

url = None
for _ in range(45):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(f"{WORK_DIR}/tunnel.log").read())
    if m:
        url = m.group(0)
        break

print("\nOpen VisionGuard:", url or "no link yet - check tunnel.log")
print("API docs        :", f"{url}/docs" if url else "")

**Notes**

- The link stops working when the Kaggle session ends. Re-run cell 6 for a new link if it changes.
- Test with images from the MVTec AD dataset (add it through *Add Input*), since the models were trained on it.
- Server output is in `visionguard/backend/server.log`.